In [1]:
#| default_exp models

In [2]:
#| export
import torch
import torch.nn as nn


In [3]:
#| export
def f_net(ni,nf,nfs=(256,512,1024,512,256,128)):
  layers = []
  layers.append(nn.Linear(ni, nfs[0]))
  layers.append(nn.SiLU())
  for i in range(len(nfs) - 1):
    layers.append(nn.Linear(nfs[i], nfs[i+1]))
    layers.append(nn.SiLU())
  layers.append(nn.Linear(nfs[-1], nf))

  return nn.Sequential(*layers)

In [4]:
#| export
class FiLMLayer(nn.Module):
    def __init__(self, hidden_dim, cond_dim):
        super().__init__()
        self.scale = nn.Linear(cond_dim, hidden_dim)
        self.shift = nn.Linear(cond_dim, hidden_dim)
        nn.init.zeros_(self.scale.weight)   
        nn.init.zeros_(self.shift.weight)   
        nn.init.ones_(self.scale.bias)      

    def forward(self, x, cond_emb):
        scale = self.scale(cond_emb)
        shift = self.shift(cond_emb)
        scale = 1 + torch.tanh(scale - 1)
        return x * scale + shift